### Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [2]:
df = pd.read_excel("data/Data_FE.xlsx", sheet_name="25 Size and BEME portfolios")
df.head()

,Unnamed: 0,Average,Value,Weighted,Returns,--,Monthly,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,Size,Small,Small,Small,Small,Small,2,2.00,2.00,2.00,...,4,4.00,4.00,4.00,4,Big,Big,Big,Big,Big
1,BE/ME,Low,2,3,4,High,Low,2.00,3.00,4.00,...,Low,2.00,3.00,4.00,High,Low,2,3,4,High
2,193601,26.94,15.49,21.82,14.56,33.28,16.43,12.49,9.49,10.27,...,2.13,6.87,8.92,8.35,6.17,2.55,6.36,8.62,14.84,16.29
3,193602,9.46,12.78,6.77,10.02,9,1.7,6.71,5.61,8.92,...,2.59,4.04,6.14,4.80,8.43,1.88,1.18,3.99,3.38,3.75
4,193603,9.46,1.38,5.56,3.01,1.24,-0.37,1.35,3.33,-1.11,...,-0.46,-0.40,3.19,1.52,-4.36,3.58,1.27,-1.82,1.26,-3.56


In [3]:
# Set rows 1 and 2 as MultiIndex header
df.columns = pd.MultiIndex.from_arrays([df.iloc[0], df.iloc[1]])
df = df.iloc[2:].reset_index(drop=True)

# Convert Date column to Year-Month
# Get the column name first
date_col = df.columns[0]

df[date_col] = pd.to_datetime(df[date_col].astype(int), format="%Y%m")

df.head()

0       Size  Small                                  2                       \
1      BE/ME    Low      2      3      4   High    Low      2      3      4   
0 1936-01-01  26.94  15.49  21.82  14.56  33.28  16.43  12.49   9.49  10.27   
1 1936-02-01   9.46  12.78   6.77  10.02      9    1.7   6.71   5.61   8.92   
2 1936-03-01   9.46   1.38   5.56   3.01   1.24  -0.37   1.35   3.33  -1.11   
3 1936-04-01  -28.6 -29.05 -12.37 -14.03 -21.72 -19.41 -13.35 -15.93 -16.43   
4 1936-05-01   1.81   8.62   1.89   10.6    5.9   5.21   4.59   7.57   5.84   

0  ...     4                              Big                            
1  ...   Low     2      3      4   High   Low     2     3      4   High  
0  ...  2.13  6.87   8.92   8.35   6.17  2.55  6.36  8.62  14.84  16.29  
1  ...  2.59  4.04   6.14   4.80   8.43  1.88  1.18  3.99   3.38   3.75  
2  ... -0.46 -0.40   3.19   1.52  -4.36  3.58  1.27 -1.82   1.26  -3.56  
3  ... -8.31 -9.00 -11.90 -12.09 -12.46 -6.17 -8.26 -7.05 -10.57     -8  
4  ...  5.12  1.79   4.54   5.98   8.18  4.78   5.2  4.71   6.25   8.39  

[5 rows x 26 columns]

In [6]:
# df: rows = date, columns = 25 portfolios
# reshape into (date, size, beme)

# Example indexing idea:
# size groups: 0=Small ... 4=Big
# beme groups: 0=Low ... 4=High

smb = []
hml = []

for date in df.index:
    row = df.loc[date]

    # SMALL vs BIG
    small = row.loc['Small']   # all BE/ME for Small
    big   = row.loc['Big']     # all BE/ME for Big
    small_avg = small.mean()
    big_avg   = big.mean()

    # HIGH vs LOW BE/ME
    low  = row.xs('Low', level=1)
    high = row.xs('High', level=1)
    low_avg  = low.mean()
    high_avg = high.mean()

    smb.append(small_avg - big_avg)
    hml.append(high_avg - low_avg)

factors = pd.DataFrame({"SMB": smb,"HML": hml}, index=df.index)
factors.head()


,SMB,HML
0,12.686,8.976
1,6.770,2.966
2,3.984,-3.546
3,-13.144,0.488
4,-0.102,4.714


In [7]:
print(factors.describe())

              SMB         HML
count  870.000000  870.000000
mean     0.213545    0.556982
std      4.631404    3.498691
min    -20.220000  -22.042000
25%     -2.323500   -1.299000
50%     -0.239000    0.389000
75%      2.321500    2.217000
max     39.110000   26.108000


## Fama-MacBeth Corss-sectional Approach

$$r_{it}=a_i+\beta_{1i}proxy_t^{size}+\beta_{2i}proxy_t^{beme}+\beta_{3i}Market_t+\epsilon_{it}$$

### Linear Regression using Statsmodels package

In [ ]:
X = factors[["MKT_RF", "SMB", "HML"]]
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())
results = sm.OLS()

## Macroeconomics Factor Model

In [ ]:
df2 = pd.read_excel("data/Data_FE.xlsx", sheet_name="Macroeconomic factors")

# Convert Date column to Year-Month
df2["Date"] = pd.to_datetime(df2["Date"].astype(int), format="%Y%m")
df2.head()